In [11]:
import numpy as np

# =====================================================================
# UPDATED TIMELINE PARAMETERS
# =====================================================================
T = 30.0       # Total time horizon = 30 min
nsteps = 30    # N = 30 discrete time steps (resulting in dt = 1.0 min)

# =====================================================================
# TARGET SETPOINTS (SP) FROM TABLE 2
# =====================================================================
SP = {
    'CV': [1.00 for _ in range(nsteps)],
    'Ln': [15.00 for _ in range(nsteps)]
}

# =====================================================================
# UPDATED ACTION SPACE (Incremental control bounds)
# =====================================================================
action_space = {
    'low': np.array([-1.0]),   # Minimum delta T step allowed per minute
    'high': np.array([1.0])    # Maximum delta T step allowed per minute
}

# =====================================================================
# OBSERVATION SPACE (State Bounds from Table 2 + Setpoints)
# =====================================================================
# Order: [mu0, mu1, mu2, mu3, C, CV, Ln, CV_SP, Ln_SP]
observation_space = {
    'low' : np.array([0.0, 0.0, 0.0, 0.0, 0.00, 0.00, 0.00, 0.00, 0.00]),
    'high' : np.array([1.0e20, 1.0e20, 1.0e20, 1.0e20, 0.50, 2.00, 20.00, 2.00, 20.00])  
}

# =====================================================================
# ENVIRONMENT PARAMETERS SETUP
# =====================================================================
env_params = {
    'N': nsteps, 
    'tsim': T, 
    'SP': SP, 
    'o_space': observation_space, 
    'a_space': action_space,
    
    # Initial conditions (x0) including the initial setpoints
    'x0': np.array([1.50e3, 2.30e4, 1.80e6, 2.50e8, 0.16, 1.00, 15.00, 1.00, 15.00]),
    
    'r_scale': {
        'CV': 1e1,
        'Ln': 1e0
    },
    
    'model': 'crystallization', 
    
    'normalise_a': True, 
    'normalise_o': True, 
    'noise': True, 
    'integration_method': 'jax', 
    'noise_percentage': 0.01, 
}

## **Normalized with Random Actions**

In [3]:
import pandas as pd
from pcgym import make_env
from tqdm import trange

# 1. Use your FULL 9-dimensional environment so you can capture the true mu3!
base_env = make_env(env_params)

# Storage lists for your dataset
dataset = []
num_episodes = 5000  # Run multiple batches to collect diverse data

for episode in trange(num_episodes):
    obs, info = base_env.reset()
    done = False
    truncated = False
    
    while not (done or truncated):
        # 2. Extract the physical layout of the current observation
        # obs order: [mu0, mu1, mu2, mu3, C, CV, Ln, CV_SP, Ln_SP]
        
        mu0, mu1, mu2, mu3, C, CV, Ln, CV_SP, Ln_SP = obs
        
        # 3. Save your measurable inputs (What the Soft Sensor will see)
        # You can choose to include mu0-mu2, or stick purely to easy measurements like C, CV, Ln.
        current_inputs = [mu0, mu1, mu2, C, CV, Ln, CV_SP, Ln_SP, mu3]
        
        dataset.append(current_inputs)
        
        # 5. Take an exploratory random action to gather diverse state space data
        random_action = base_env.action_space.sample() 
        obs, reward, done, truncated, info = base_env.step(random_action)

# 6. Convert to a Pandas DataFrame for easy Machine Learning training
feature_names = ['mu0', 'mu1', 'mu2', 'C', 'CV', 'Ln', 'CV_SP', 'Ln_SP', 'mu3']
dataset = pd.DataFrame(dataset, columns=feature_names)

print("Dataset ready!")
print("X Shape (Features):", dataset.shape)
print('-'*30)
print(dataset)
dataset.to_csv(f'cryst-{num_episodes}ep.csv', index=False)
print(f'Dataset is saved successfully with {num_episodes} episodes!!!')

/Users/phattharapongduangkham/Documents/crystallization-partial-drl-ml/.venv/lib/python3.12/site-packages/gymnasium/spaces/box.py:231: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/phattharapongduangkham/Documents/crystallization-partial-drl-ml/.venv/lib/python3.12/site-packages/gymnasium/spaces/box.py:297: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
  0%|          | 0/5000 [00:00<?, ?it/s]

100%|██████████| 5000/5000 [1:16:05<00:00,  1.10it/s]  


Dataset ready!
X Shape (Features): (145000, 9)
------------------------------
        mu0  mu1  mu2         C        CV        Ln  CV_SP  Ln_SP  mu3
0      -1.0 -1.0 -1.0 -0.360000  0.000000  0.500000    0.0    0.5 -1.0
1      -1.0 -1.0 -1.0 -0.363845 -0.917765  1.725569    0.0    0.5 -1.0
2      -1.0 -1.0 -1.0 -0.362031 -1.154758  2.631652    0.0    0.5 -1.0
3      -1.0 -1.0 -1.0 -0.361980 -1.172934  2.943233    0.0    0.5 -1.0
4      -1.0 -1.0 -1.0 -0.382700 -1.095852  2.977309    0.0    0.5 -1.0
...     ...  ...  ...       ...       ...       ...    ...    ...  ...
144995 -1.0 -1.0 -1.0 -0.659736 -0.572898  0.549136    0.0    0.5 -1.0
144996 -1.0 -1.0 -1.0 -0.667695 -0.557340  0.515029    0.0    0.5 -1.0
144997 -1.0 -1.0 -1.0 -0.674181 -0.552504  0.505917    0.0    0.5 -1.0
144998 -1.0 -1.0 -1.0 -0.673231 -0.528849  0.471441    0.0    0.5 -1.0
144999 -1.0 -1.0 -1.0 -0.682444 -0.523362  0.440253    0.0    0.5 -1.0

[145000 rows x 9 columns]
Dataset is saved successfully with 5000 epi

## **Unnormalized with Random Actions**

In [14]:
from stable_baselines3 import PPO
from pcgym import make_env

# 1. Use your FULL 9-dimensional environment
base_env = make_env(env_params)
# 2. Load trained PPO model
ppo_model = PPO.load("../drl-compare/ppo-default", env=base_env)

/Users/phattharapongduangkham/Documents/crystallization-partial-drl-ml/.venv/lib/python3.12/site-packages/gymnasium/spaces/box.py:231: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/phattharapongduangkham/Documents/crystallization-partial-drl-ml/.venv/lib/python3.12/site-packages/gymnasium/spaces/box.py:297: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [ ]:
import pandas as pd
import os
from tqdm import trange

# Settings
num_episodes = 10000  
save_every_n_episodes = 500  # Save data to disk periodically to free RAM
output_filename = f'cryst-unnormalized-{num_episodes}ep-ppo.csv'

# Delete the file if it already exists from a previous failed run
if os.path.exists(output_filename):
    os.remove(output_filename)

feature_names = ['mu0', 'mu1', 'mu2', 'C', 'CV', 'Ln', 'CV_SP', 'Ln_SP', 'mu3']
dataset = []

print("Starting data collection...")

for episode in trange(num_episodes):
    obs, info = base_env.reset()
    done = False
    truncated = False
    
    while not (done or truncated):
        # obs order: [mu0, mu1, mu2, mu3, C, CV, Ln, CV_SP, Ln_SP]
        raw_state = base_env.state
        
        mu0 = raw_state[0]
        mu1 = raw_state[1]
        mu2 = raw_state[2]
        mu3 = raw_state[3]
        C   = raw_state[4]
        CV  = raw_state[5]
        Ln  = raw_state[6]
        
        CV_SP = base_env.SP['CV'][base_env.t] if hasattr(base_env, 't') else 1.0
        Ln_SP = base_env.SP['Ln'][base_env.t] if hasattr(base_env, 't') else 15.0

        current_inputs = [mu0, mu1, mu2, C, CV, Ln, CV_SP, Ln_SP, mu3]
        dataset.append(current_inputs)
        
        # Take exploratory action
        # random_action = base_env.action_space.sample() 
        # Take default step
        action, _ = ppo_model.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = base_env.step(action)

    # --- MEMORY MANAGEMENT BLOCK ---
    if (episode + 1) % save_every_n_episodes == 0:
        # Convert the current chunk to a DataFrame
        df_chunk = pd.DataFrame(dataset, columns=feature_names)
        
        # Append to CSV (write header only on the very first write)
        header_condition = not os.path.exists(output_filename)
        df_chunk.to_csv(output_filename, mode='a', index=False, header=header_condition)
        
        print(f"Saved episodes up to {episode + 1}/{num_episodes}. Clearing RAM.")
        
        # Wipe the list completely to give the RAM back to macOS
        dataset.clear() 

# Save any remaining episodes that didn't hit the exact batch limit
if len(dataset) > 0:
    df_chunk = pd.DataFrame(dataset, columns=feature_names)
    df_chunk.to_csv(output_filename, mode='a', index=False, header=not os.path.exists(output_filename))

print(f'\nDataset is saved successfully to {output_filename}!!!')

Starting data collection...


  0%|          | 0/10000 [00:00<?, ?it/s]/Users/phattharapongduangkham/Documents/crystallization-partial-drl-ml/.venv/lib/python3.12/site-packages/gymnasium/spaces/box.py:231: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/phattharapongduangkham/Documents/crystallization-partial-drl-ml/.venv/lib/python3.12/site-packages/gymnasium/spaces/box.py:297: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
  5%|▌         | 500/10000 [03:40<1:03:03,  2.51it/s] 

Saved episodes up to 500/10000. Clearing RAM.


/Users/phattharapongduangkham/Documents/crystallization-partial-drl-ml/.venv/lib/python3.12/site-packages/gymnasium/spaces/box.py:231: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/phattharapongduangkham/Documents/crystallization-partial-drl-ml/.venv/lib/python3.12/site-packages/gymnasium/spaces/box.py:297: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
 10%|█         | 1000/10000 [07:55<1:35:33,  1.57it/s]

Saved episodes up to 1000/10000. Clearing RAM.


/Users/phattharapongduangkham/Documents/crystallization-partial-drl-ml/.venv/lib/python3.12/site-packages/gymnasium/spaces/box.py:231: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/phattharapongduangkham/Documents/crystallization-partial-drl-ml/.venv/lib/python3.12/site-packages/gymnasium/spaces/box.py:297: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
 15%|█▌        | 1500/10000 [14:54<1:09:55,  2.03it/s] 

Saved episodes up to 1500/10000. Clearing RAM.


/Users/phattharapongduangkham/Documents/crystallization-partial-drl-ml/.venv/lib/python3.12/site-packages/gymnasium/spaces/box.py:231: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/phattharapongduangkham/Documents/crystallization-partial-drl-ml/.venv/lib/python3.12/site-packages/gymnasium/spaces/box.py:297: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
 19%|█▉        | 1881/10000 [19:54<1:42:19,  1.32it/s]

In [10]:
import pandas as pd

df = pd.read_csv('./cryst-unnormalized-500ep.csv')
df.describe()

,mu0,mu1,mu2,C,CV,Ln,CV_SP,Ln_SP,mu3
count,14500.000000,1.450000e+04,1.450000e+04,14500.000000,14500.000000,14500.000000,14500.0,14500.0,1.450000e+04
mean,149367.244580,2.628597e+06,1.379697e+08,0.114236,0.273026,22.636420,1.0,15.0,1.400405e+10
std,123696.856803,1.889281e+06,9.099956e+07,0.029421,0.224404,7.823764,0.0,0.0,8.842213e+09
min,1500.000000,2.300000e+04,1.800000e+06,0.080174,-0.176044,14.153557,1.0,15.0,2.500000e+08
25%,17948.882812,5.533971e+05,4.029799e+07,0.086590,0.166873,16.021309,1.0,15.0,4.691721e+09
50%,140507.640625,2.940309e+06,1.644410e+08,0.103763,0.286435,19.768794,1.0,15.0,1.715163e+10
75%,266747.125000,4.488743e+06,2.255179e+08,0.145221,0.405893,27.548179,1.0,15.0,2.231258e+10
max,354127.156250,5.245782e+06,2.506822e+08,0.160000,1.000000,40.056782,1.0,15.0,2.424106e+10


In [4]:
!pwd

/Users/phattharapongduangkham/Documents/crystallization-partial-drl-ml/dataset-generation
